# Cancer Mutation Detection - Pipeline Optimisé

**Prédiction de Mutations Génétiques Pathogènes au Cancer**

Ce notebook exécute le pipeline ML optimisé avec **2 modèles** :
1. **XGBoost** avec GridSearchCV (5-fold CV)
2. **Neural Network Simple** avec Early Stopping + ReduceLROnPlateau

### Améliorations
- **XGBoost** : GridSearchCV sur max_depth, learning_rate, n_estimators + scale_pos_weight
- **NN Simple** : Early Stopping (patience=15), ReduceLROnPlateau, test batch_size [16, 32, 64], L2 regularization

In [1]:
# Configuration des imports
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Fixer le seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Imports configurés")
print(f"✓ Seed: {RANDOM_STATE}")

✓ Imports configurés
✓ Seed: 42


---

## ÉTAPE 1: Chargement des Données

In [2]:
from data_loader import load_clinvar_data, explore_data, get_features_info

# Charger le dataset
df = load_clinvar_data('../data/clinvar_conflicting.csv', nrows=None)

# Exploration rapide
stats = explore_data(df)

# Afficher les features sélectionnées
features_info = get_features_info()
print("\n" + "="*60)
print("FEATURES SÉLECTIONNÉES")
print("="*60)
for feat, info in features_info.items():
    print(f"\n{feat}:")
    print(f"  Description: {info['description']}")
    print(f"  Importance: {info['importance']}")

Chargement des données depuis: ../data/clinvar_conflicting.csv
Dataset chargé: 65,188 lignes, 46 colonnes

EXPLORATION DES DONNÉES

Dimensions: 65,188 lignes × 46 colonnes
Mémoire utilisée: 125.27 MB

Distribution de la cible (CLNSIGINCL):
CLNSIGINCL
NaN                             65021
424754:Likely_pathogenic            2
30118:risk_factor                   2
60702:Pathogenic                    2
236068:Pathogenic                   2
                                ...  
140570:Pathogenic                   1
3620:Benign                         1
424813:Pathogenic                   1
10382:other                         1
10361:Pathogenic|10382:other        1
Name: count, Length: 138, dtype: int64

FEATURES SÉLECTIONNÉES

CADD_PHRED:
  Description: Score de pathogénicité combiné (0-100)
  Importance: Très élevée - Score expert #1

CADD_RAW:
  Description: Version brute du score CADD
  Importance: Élevée - Complète CADD_PHRED

SIFT:
  Description: Score d'impact sur la protéine (0-1)
 

---

## ÉTAPE 2: Prétraitement

In [3]:
from preprocessing import clean_clinvar_data, prepare_train_test_data

# Nettoyer les données
df_clean, features = clean_clinvar_data(df)

# Préparer train/test
X_train, X_test, y_train, y_test, scaler, feature_names = prepare_train_test_data(
    df_clean, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)

print(f"\n✓ Prétraitement terminé")
print(f"  Features: {feature_names}")
print(f"  Train: {X_train.shape[0]:,} samples")
print(f"  Test: {X_test.shape[0]:,} samples")


NETTOYAGE DES DONNÉES
Données brutes: 65,188 lignes

Distribution de la cible (CLASS):
  Benign (0):     48,754 (74.8%)
  Pathogenic (1): 16,434 (25.2%)

SIFT encodé: {0.0: 12561, 1.0: 12275}
PolyPhen encodé: {0.0: 13329, 1.0: 7531, 0.5: 3932}
  ➕ SIFT_missing: indicateur ajouté (61.9% de NaN)
  ➕ PolyPhen_missing: indicateur ajouté (62.0% de NaN)
  ➕ BLOSUM62_missing: indicateur ajouté (60.7% de NaN)

  3 indicateurs de manquantes ajoutés: ['SIFT_missing', 'PolyPhen_missing', 'BLOSUM62_missing']
  CADD_PHRED: 1,092 valeurs manquantes → médiane (14.0900)
  CADD_RAW: 1,092 valeurs manquantes → médiane (1.6429)
  SIFT: 40,352 valeurs manquantes → médiane (0.0000)
  PolyPhen: 40,396 valeurs manquantes → médiane (0.0000)
  BLOSUM62: 39,595 valeurs manquantes → médiane (-1.0000)

Dataset final: 65,188 lignes × 11 colonnes
Features originales: ['CADD_PHRED', 'CADD_RAW', 'SIFT', 'PolyPhen', 'AF_EXAC', 'AF_TGP', 'BLOSUM62']
Indicateurs manquantes: ['SIFT_missing', 'PolyPhen_missing', 'BLOSUM6

---

## ÉTAPE 3: Entraînement Optimisé des 2 Modèles

### 3A: XGBoost avec GridSearchCV (5-fold)

In [4]:
from train_optimized import train_xgboost_optimized

# Entraîner XGBoost avec GridSearchCV
xgb_model, xgb_grid_results = train_xgboost_optimized(
    X_train, y_train, 
    random_state=RANDOM_STATE
)

print("\nRésultats GridSearchCV détaillés:")
print(f"  Meilleur max_depth: {xgb_grid_results['best_params']['max_depth']}")
print(f"  Meilleur learning_rate: {xgb_grid_results['best_params']['learning_rate']}")
print(f"  Meilleur n_estimators: {xgb_grid_results['best_params']['n_estimators']}")
print(f"  Meilleur ROC-AUC (CV 5-fold): {xgb_grid_results['best_score']:.4f}")

I0000 00:00:1776280281.212770   15926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776280281.221247   15926 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776280281.983823   15926 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776280286.108651   15926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0


ENTRAÎNEMENT XGBOOST OPTIMISÉ (GridSearchCV)
Distribution des classes: 39003 négatifs, 13147 positifs
scale_pos_weight: 2.97

Grille de recherche: 27 combinaisons × 5 folds = 135 entraînements
  max_depth: [4, 6, 8]
  learning_rate: [0.01, 0.05, 0.1]
  n_estimators: [100, 200, 300]

Début du GridSearchCV...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

✓ GridSearchCV terminé en 42.0s

────────────────────────────────────────
MEILLEURS HYPERPARAMÈTRES XGBOOST:
────────────────────────────────────────
  learning_rate: 0.05
  max_depth: 6
  n_estimators: 100

  Meilleur ROC-AUC (CV): 0.7340
────────────────────────────────────────

Résultats GridSearchCV détaillés:
  Meilleur max_depth: 6
  Meilleur learning_rate: 0.05
  Meilleur n_estimators: 100
  Meilleur ROC-AUC (CV 5-fold): 0.7340


### 3B: Neural Network Simple Optimisé

In [4]:
from train_optimized import train_nn_simple_optimized

# Entraîner NN Simple avec Early Stopping + ReduceLROnPlateau
nn_simple, nn_history, nn_results = train_nn_simple_optimized(
    X_train, y_train,
    batch_sizes_to_test=[16, 32, 64],
    epochs=150,
    random_state=RANDOM_STATE
)

print("\nRésultats Tuning NN Simple:")
print(f"  Meilleur batch_size: {nn_results['best_batch_size']}")
print(f"  Meilleur val_AUC: {nn_results['best_val_auc']:.4f}")
print(f"  Epochs effectifs: {len(nn_history.history['loss'])}")


I0000 00:00:1776283886.657606   47859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776283886.658229   47859 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776283886.686089   47859 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776283888.901520   47859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0


ENTRAÎNEMENT NEURAL NETWORK SIMPLE OPTIMISÉ
Architecture: 64 → 32 → 1 (avec L2=0.0005)
Epochs max: 150
Batch sizes à tester: [16, 32, 64]
Early Stopping patience: 15
ReduceLROnPlateau: factor=0.5, patience=5

Class weights (sqrt-dampened):
  Benign (0):     0.735  (balanced brut: 0.669)
  Pathogenic (1): 1.265  (balanced brut: 1.983)
  → Ratio effectif: 1.72x (vs 2.97x brut)

────────────────────────────────────────
Test avec batch_size = 16
────────────────────────────────────────


E0000 00:00:1776283890.584253   47859 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776283890.585586   47935 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1776283890.598570   47859 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/150
2608/2608 - 10s - 4ms/step - auc: 0.5933 - loss: 0.5677 - precision: 0.2820 - recall: 0.0184 - val_auc: 0.6154 - val_loss: 0.5909 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/150
2608/2608 - 8s - 3ms/step - auc: 0.6099 - loss: 0.5526 - precision: 0.3000 - recall: 0.0026 - val_auc: 0.6172 - val_loss: 0.5841 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/150
2608/2608 - 6s - 2ms/step - auc: 0.6093 - loss: 0.5477 - precision: 0.3333 - recall: 9.4931e-05 - val_auc: 0.6198 - val_loss: 0.5795 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/150
2608/2608 - 6s - 2ms/step - auc: 0.6113 - loss: 0.5452 - precision: 0.5000 - recall: 9.4931e-05 - val_auc: 0.6161 - val_loss: 0.5779 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/150
2608/2608 - 6s - 2ms/step - auc: 0.6141 - loss: 0.5443 - precision: 0.4000 - recall: 1.8986e-04 - val_auc: 

### Sauvegarde des modèles optimisés

In [ ]:
from train_optimized import save_optimized_models

save_optimized_models(
    xgb_model, nn_simple, nn_history,
    xgb_grid_results, nn_results,
    output_dir='../models'
)

---

## ÉTAPE 4: Évaluation et Comparaison

In [ ]:
from evaluate_optimized import evaluate_all_optimized

# Évaluer les 2 modèles et générer les visualisations
eval_results = evaluate_all_optimized(
    xgb_model, nn_simple,
    X_test, y_test, feature_names,
    nn_simple_history=nn_history.history,
    xgb_grid_results=xgb_grid_results,
    nn_results=nn_results,
    output_dir='../results'
)

---

## ÉTAPE 5: Prédiction sur Cas Réels

In [ ]:
# Exemple de mutation pathogène (Cancer)
example_pathogenic = np.array([[
    28.0,      # CADD_PHRED (élevé = pathogène)
    3.5,       # CADD_RAW
    0.01,      # SIFT (bas = délétère)
    0.95,      # PolyPhen (élevé = dangereux)
    0.0001,    # AF_EXAC (rare)
    0.0002,    # AF_TGP (rare)
    -3.0       # BLOSUM62 (négatif = grave)
]])

# Exemple de mutation bénigne
example_benign = np.array([[
    5.0,       # CADD_PHRED (faible)
    0.5,       # CADD_RAW
    0.8,       # SIFT (élevé = toléré)
    0.1,       # PolyPhen (faible)
    0.35,      # AF_EXAC (commun)
    0.40,      # AF_TGP (commun)
    2.0        # BLOSUM62 (positif)
]])

# Normaliser
example_pathogenic_scaled = scaler.transform(example_pathogenic)
example_benign_scaled = scaler.transform(example_benign)

for name, example, example_scaled in [
    ("MUTATION PATHOGÈNE (Cancer)", example_pathogenic, example_pathogenic_scaled),
    ("MUTATION BÉNIGNE (Inoffensive)", example_benign, example_benign_scaled)
]:
    print(f"\n{name}:")
    print(f"  Features: CADD_PHRED={example[0][0]:.1f}, SIFT={example[0][2]:.2f}, PolyPhen={example[0][3]:.2f}")
    
    pred_xgb = xgb_model.predict_proba(example_scaled)[0, 1]
    pred_nn_simple = nn_simple.predict(example_scaled, verbose=0)[0, 0]
    pred_avg = (pred_xgb + pred_nn_simple) / 2
    
    print(f"  Probabilité Pathogène:")
    print(f"    XGBoost:    {pred_xgb:.1%}")
    print(f"    NN Simple:  {pred_nn_simple:.1%}")
    print(f"    Moyenne:    {pred_avg:.1%}")
    
    verdict = "PATHOGÈNE ⚠️" if pred_avg > 0.5 else "BÉNIGNE ✓"
    print(f"  → VERDICT: {verdict}")

---

## Résumé Final

In [ ]:
# Trouver le meilleur modèle
roc_aucs = [r['roc_auc'] for r in eval_results]
best_idx = np.argmax(roc_aucs)
best_model = eval_results[best_idx]

print("\n" + "="*60)
print("RÉSUMÉ FINAL")
print("="*60)
print(f"\n🏆 Meilleur modèle: {best_model['model_name']}")
print(f"   ROC-AUC: {best_model['roc_auc']:.4f}")

print(f"\n📊 Meilleurs hyperparamètres XGBoost (GridSearchCV):")
for param, value in xgb_grid_results['best_params'].items():
    print(f"   {param}: {value}")
print(f"   CV ROC-AUC: {xgb_grid_results['best_score']:.4f}")

print(f"\n🧠 Meilleurs hyperparamètres NN Simple:")
print(f"   batch_size: {nn_results['best_batch_size']}")
print(f"   L2 regularization: 0.001")
print(f"   Early Stopping patience: 15")
print(f"   val_AUC: {nn_results['best_val_auc']:.4f}")

if best_model['roc_auc'] >= 0.75:
    print(f"\n✅ OBJECTIF ATTEINT: ROC-AUC ≥ 75% ({best_model['roc_auc']:.1%})")
else:
    print(f"\n⚠️  Objectif non atteint: ROC-AUC < 75% ({best_model['roc_auc']:.1%})")

print(f"\n📁 Résultats sauvegardés dans: ../results/")
print("="*60)

---

## Visualisations

In [ ]:
# Afficher les images générées
from IPython.display import Image, display

print("\n📊 Courbes ROC (XGBoost vs NN Simple):")
display(Image(filename='../results/roc_curves.png'))

print("\n📊 Matrices de Confusion:")
display(Image(filename='../results/confusion_matrices.png'))

print("\n📊 Feature Importance (XGBoost):")
display(Image(filename='../results/feature_importance.png'))

print("\n📊 Historique d'entraînement (NN Simple):")
display(Image(filename='../results/training_history.png'))

---

## Tableau Comparatif Final

In [ ]:
# Afficher le tableau comparatif
comparison_df = pd.read_csv('../results/model_comparison.csv')
comparison_df